In [1]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
from scipy.optimize import minimize
from statsmodels.formula.api import glm
from statsmodels.genmod.families import Poisson
import pickle
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\features.csv")
df['date'] = pd.to_datetime(df['date'])

with open(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\home_model.pkl", 'rb') as f:
    home_model = pickle.load(f)

with open(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\away_model.pkl", 'rb') as f:
    away_model = pickle.load(f)

print(f"Data loaded: {df.shape}")
print("Models loaded!")

Data loaded: (32101, 19)
Models loaded!


In [3]:
# Prepare features
model_df = df.dropna(subset=['home_elo', 'away_elo', 'elo_diff', 'fifa_rank_diff']).copy()
model_df['neutral'] = model_df['neutral'].astype(int)

# Get expected goals for every match from our trained models
features = model_df[['home_elo', 'away_elo', 'elo_diff', 'neutral', 'tournament_weight', 'fifa_rank_diff']]

model_df['home_xg'] = home_model.predict(features)
model_df['away_xg'] = away_model.predict(features)

print(f"Expected goals generated for {len(model_df)} matches")
print(f"\nAverage home xG: {model_df['home_xg'].mean():.3f}")
print(f"Average away xG: {model_df['away_xg'].mean():.3f}")

Expected goals generated for 32101 matches

Average home xG: 1.651
Average away xG: 1.108


In [4]:
def dixon_coles_tau(home_goals, away_goals, home_xg, away_xg, rho):
    if home_goals == 0 and away_goals == 0:
        return 1 - home_xg * away_xg * rho
    elif home_goals == 1 and away_goals == 0:
        return 1 + away_xg * rho
    elif home_goals == 0 and away_goals == 1:
        return 1 + home_xg * rho
    elif home_goals == 1 and away_goals == 1:
        return 1 - rho
    else:
        return 1.0

def log_likelihood(rho, data):
    total = 0
    for _, row in data.iterrows():
        h = int(row['home_score'])
        a = int(row['away_score'])
        lh = row['home_xg']
        la = row['away_xg']
        tau = dixon_coles_tau(h, a, lh, la, rho)
        p = poisson.pmf(h, lh) * poisson.pmf(a, la) * tau
        if p > 0:
            total += np.log(p)
    return -total

print("Estimating rho — this will take a minute...")
result = minimize(
    log_likelihood,
    x0=[-0.1],
    args=(model_df,),
    method='Nelder-Mead',
    options={'xatol': 0.001, 'fatol': 0.001}
)

rho = result.x[0]
print(f"Estimated rho: {rho:.4f}")

Estimating rho — this will take a minute...
Estimated rho: -0.0513


In [5]:
def match_probabilities_dc(home_xg, away_xg, rho=-0.0513, max_goals=10):
    home_win, draw, away_win = 0, 0, 0
    
    for i in range(max_goals):
        for j in range(max_goals):
            tau = dixon_coles_tau(i, j, home_xg, away_xg, rho)
            p = poisson.pmf(i, home_xg) * poisson.pmf(j, away_xg) * tau
            if i > j:
                home_win += p
            elif i == j:
                draw += p
            else:
                away_win += p
    
    return home_win, draw, away_win

# Compare standard vs Dixon-Coles on Spain vs Argentina
home_xg, away_xg = 1.38, 0.92

# Standard
from scipy.stats import poisson as sp
hw_std = sum(sp.pmf(i, home_xg) * sp.pmf(j, away_xg) 
             for i in range(10) for j in range(10) if i > j)
d_std  = sum(sp.pmf(i, home_xg) * sp.pmf(j, away_xg) 
             for i in range(10) for j in range(10) if i == j)
aw_std = sum(sp.pmf(i, home_xg) * sp.pmf(j, away_xg) 
             for i in range(10) for j in range(10) if i < j)

hw_dc, d_dc, aw_dc = match_probabilities_dc(home_xg, away_xg)

print("Spain vs Argentina — Standard vs Dixon-Coles:")
print(f"{'':20} {'Standard':>10} {'Dixon-Coles':>12} {'Difference':>12}")
print("-" * 55)
print(f"{'Spain win':20} {hw_std*100:>9.1f}% {hw_dc*100:>11.1f}% {(hw_dc-hw_std)*100:>+11.1f}%")
print(f"{'Draw':20} {d_std*100:>9.1f}% {d_dc*100:>11.1f}% {(d_dc-d_std)*100:>+11.1f}%")
print(f"{'Argentina win':20} {aw_std*100:>9.1f}% {aw_dc*100:>11.1f}% {(aw_dc-aw_std)*100:>+11.1f}%")

Spain vs Argentina — Standard vs Dixon-Coles:
                       Standard  Dixon-Coles   Difference
-------------------------------------------------------
Spain win                 47.5%        46.9%        -0.7%
Draw                      27.4%        28.7%        +1.3%
Argentina win             25.0%        24.4%        -0.7%


In [6]:
elo_all = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\elo_all_teams.csv")
elo_ratings_bt = dict(zip(elo_all['team'], elo_all['elo']))

# Reload backtest data
df_raw = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\results_clean.csv")
df_raw['date'] = pd.to_datetime(df_raw['date'])

train_df = df_raw[df_raw['date'] < '2022-11-20'].copy()
test_df = df_raw[
    (df_raw['date'] >= '2022-11-20') &
    (df_raw['tournament'] == 'FIFA World Cup')
].copy()

# Rebuild Elo on training data
STARTING_ELO = 1500
HOME_ADVANTAGE = 100
elo_ratings = {}

def get_elo(team):
    if team not in elo_ratings:
        elo_ratings[team] = STARTING_ELO
    return elo_ratings[team]

def expected_score(ra, rb):
    return 1 / (1 + 10 ** ((rb - ra) / 400))

def update_elo(home_team, away_team, home_score, away_score, tournament_weight, neutral, date):
    he = get_elo(home_team)
    ae = get_elo(away_team)
    ha = he if neutral else he + HOME_ADVANTAGE
    hexp = expected_score(ha, ae)
    aexp = expected_score(ae, ha)
    if home_score > away_score: hact, aact = 1, 0
    elif home_score < away_score: hact, aact = 0, 1
    else: hact, aact = 0.5, 0.5
    year = pd.to_datetime(date).year
    bk = 35 if year >= 2022 else 30 if year >= 2018 else 20
    k = bk * tournament_weight
    elo_ratings[home_team] = he + k * (hact - hexp)
    elo_ratings[away_team] = ae + k * (aact - aexp)

train_df = train_df.sort_values('date').reset_index(drop=True)
for _, row in train_df.iterrows():
    update_elo(row['home_team'], row['away_team'], row['home_score'],
               row['away_score'], row['tournament_weight'], row['neutral'], row['date'])

# Retrain models
train_df['neutral'] = train_df['neutral'].astype(int)
elo_df_bt = pd.DataFrame(list(elo_ratings.items()), columns=['team', 'elo'])
train_features = train_df.merge(elo_df_bt, left_on='home_team', right_on='team', how='left')
train_features = train_features.rename(columns={'elo': 'home_elo'}).drop(columns='team')
train_features = train_features.merge(elo_df_bt, left_on='away_team', right_on='team', how='left')
train_features = train_features.rename(columns={'elo': 'away_elo'}).drop(columns='team')
train_features['elo_diff'] = train_features['home_elo'] - train_features['away_elo']
train_features['fifa_rank_diff'] = 0
train_features = train_features.dropna(subset=['home_elo', 'away_elo'])

home_model_bt = glm('home_score ~ home_elo + away_elo + elo_diff + neutral + tournament_weight',
                    data=train_features, family=Poisson()).fit()
away_model_bt = glm('away_score ~ home_elo + away_elo + elo_diff + neutral + tournament_weight',
                    data=train_features, family=Poisson()).fit()

print("Models retrained!")

Models retrained!


In [7]:
def predict_match_bt(home_team, away_team, neutral=True):
    home_elo = elo_ratings.get(home_team, 1500)
    away_elo = elo_ratings.get(away_team, 1500)
    match = pd.DataFrame([{
        'home_elo': home_elo,
        'away_elo': away_elo,
        'elo_diff': home_elo - away_elo,
        'neutral': int(neutral),
        'tournament_weight': 5.0,
        'fifa_rank_diff': 0
    }])
    home_xg = home_model_bt.predict(match)[0]
    away_xg = away_model_bt.predict(match)[0]
    return home_xg, away_xg

def standard_probs(home_xg, away_xg, max_goals=10):
    hw, d, aw = 0, 0, 0
    for i in range(max_goals):
        for j in range(max_goals):
            p = poisson.pmf(i, home_xg) * poisson.pmf(j, away_xg)
            if i > j: hw += p
            elif i == j: d += p
            else: aw += p
    return hw, d, aw

predictions_std = []
predictions_dc = []

for _, row in test_df.iterrows():
    home_xg, away_xg = predict_match_bt(row['home_team'], row['away_team'])

    hw_s, d_s, aw_s = standard_probs(home_xg, away_xg)
    hw_d, d_d, aw_d = match_probabilities_dc(home_xg, away_xg, rho=rho)

    if row['home_score'] > row['away_score']: actual = 'H'
    elif row['home_score'] < row['away_score']: actual = 'A'
    else: actual = 'D'

    for pred_list, hw, d, aw in [(predictions_std, hw_s, d_s, aw_s),
                                   (predictions_dc, hw_d, d_d, aw_d)]:
        predicted = 'H' if hw > d and hw > aw else ('D' if d > aw else 'A')
        
        actual_oh = {'H': 0, 'D': 0, 'A': 0}
        actual_oh[actual] = 1
        brier = (hw - actual_oh['H'])**2 + (d - actual_oh['D'])**2 + (aw - actual_oh['A'])**2
        eps = 1e-10
        ll = -(actual_oh['H']*np.log(hw+eps) + actual_oh['D']*np.log(d+eps) + actual_oh['A']*np.log(aw+eps))
        
        pred_list.append({
            'actual': actual, 'predicted': predicted,
            'brier': brier, 'log_loss': ll,
            'p_home': hw, 'p_draw': d, 'p_away': aw
        })

std_df = pd.DataFrame(predictions_std)
dc_df = pd.DataFrame(predictions_dc)

print("=== STANDARD vs DIXON-COLES ===\n")
print(f"{'Metric':<20} {'Standard':>10} {'Dixon-Coles':>12} {'Improvement':>12}")
print("-" * 55)

acc_std = (std_df['actual'] == std_df['predicted']).mean()
acc_dc  = (dc_df['actual'] == dc_df['predicted']).mean()
bs_std  = std_df['brier'].mean()
bs_dc   = dc_df['brier'].mean()
ll_std  = std_df['log_loss'].mean()
ll_dc   = dc_df['log_loss'].mean()

print(f"{'Accuracy':<20} {acc_std*100:>9.1f}% {acc_dc*100:>11.1f}% {(acc_dc-acc_std)*100:>+11.1f}%")
print(f"{'Brier Score':<20} {bs_std:>10.4f} {bs_dc:>12.4f} {(bs_dc-bs_std):>+12.4f}")
print(f"{'Log Loss':<20} {ll_std:>10.4f} {ll_dc:>12.4f} {(ll_dc-ll_std):>+12.4f}")
print(f"\nDraws predicted (Standard):    {(std_df['predicted']=='D').sum()}")
print(f"Draws predicted (Dixon-Coles): {(dc_df['predicted']=='D').sum()}")
print(f"Actual draws:                  {(std_df['actual']=='D').sum()}")

=== STANDARD vs DIXON-COLES ===

Metric                 Standard  Dixon-Coles  Improvement
-------------------------------------------------------
Accuracy                  48.4%        48.4%        +0.0%
Brier Score              0.6139       0.6146      +0.0007
Log Loss                 1.0296       1.0316      +0.0020

Draws predicted (Standard):    0
Draws predicted (Dixon-Coles): 0
Actual draws:                  15


In [8]:
def predict_outcome(hw, d, aw, draw_threshold=0.25):
    # Predict draw if draw probability is above threshold
    # AND the gap between home and away win is small
    if d >= draw_threshold and abs(hw - aw) < 0.15:
        return 'D'
    elif hw > aw:
        return 'H'
    else:
        return 'A'

# Test different thresholds
print(f"{'Threshold':<12} {'Accuracy':>10} {'Draws predicted':>16} {'Brier':>8}")
print("-" * 50)

for threshold in [0.20, 0.22, 0.24, 0.25, 0.26, 0.28, 0.30]:
    preds = []
    briers = []
    for _, row in dc_df.iterrows():
        pred = predict_outcome(row['p_home'], row['p_draw'], row['p_away'], threshold)
        preds.append(pred)
    
    pred_series = pd.Series(preds)
    acc = (pred_series == std_df['actual']).mean()
    draws = (pred_series == 'D').sum()
    
    print(f"{threshold:<12} {acc*100:>9.1f}% {draws:>16} {bs_dc:>8.4f}")

Threshold      Accuracy  Draws predicted    Brier
--------------------------------------------------
0.2               39.1%               21   0.6146
0.22              39.1%               21   0.6146
0.24              39.1%               21   0.6146
0.25              39.1%               21   0.6146
0.26              39.1%               21   0.6146
0.28              43.8%               18   0.6146
0.3               48.4%                0   0.6146


In [9]:
# Check draw rates across different tournaments in our data
draw_rates = df.groupby('tournament').apply(
    lambda x: (x['home_score'] == x['away_score']).mean()
).sort_values(ascending=False)

print("Draw rates by tournament (top 15):")
print(draw_rates.head(15).round(3).to_string())

print(f"\nOverall draw rate in training data: {(df['home_score']==df['away_score']).mean():.3f}")
print(f"2022 WC draw rate: {(test_df['home_score']==test_df['away_score']).mean():.3f}")

Draw rates by tournament (top 15):
tournament
Copa Lipton                             1.000
TIFOCO Tournament                       1.000
Copa Confraternidad                     1.000
Dakar Tournament                        1.000
Four Nations' Cup                       1.000
Al Ain International Cup                0.750
Mahinda Rajapaksa Cup                   0.571
Copa Paz del Chaco                      0.556
Cup of Ancient Civilizations            0.500
Morocco, Capital of African Football    0.500
ASEAN Championship qualification        0.500
Joe Robbie Cup                          0.500
Marianas Cup                            0.500
CONMEBOL–UEFA Cup of Champions          0.500
Tournoi de France                       0.500

Overall draw rate in training data: 0.235
2022 WC draw rate: 0.234


In [10]:
dc_params = {'rho': rho}

with open(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\dc_params.pkl", 'wb') as f:
    pickle.dump(dc_params, f)

print(f"Saved Dixon-Coles rho = {rho:.4f}")
print("\n=== NOTEBOOK 07 COMPLETE ===")
print(f"Dixon-Coles rho:     {rho:.4f}")
print(f"Improvement in accuracy:   +0.0%  (model already well-calibrated)")
print(f"Improvement in Brier:      -0.0007 (negligible)")
print(f"Conclusion: Standard Poisson is performing well for international football")
print(f"Best next step: Add more features (recent form, head-to-head, squad strength)")

Saved Dixon-Coles rho = -0.0513

=== NOTEBOOK 07 COMPLETE ===
Dixon-Coles rho:     -0.0513
Improvement in accuracy:   +0.0%  (model already well-calibrated)
Improvement in Brier:      -0.0007 (negligible)
Conclusion: Standard Poisson is performing well for international football
Best next step: Add more features (recent form, head-to-head, squad strength)
